## 1.3

Моделирование выборок распределения Лапласа

In [ ]:
import random
import math
import matplotlib.pyplot as plt
import numpy as np


def laplace(mu, teta):
  u = random.uniform(0, 1)
  if u <= 0.5:
    return mu+(teta**-1)*math.log(2*u)
  else:
    return mu-(teta**-1)*math.log(2*(1-u))

my_mu = 12.0
my_teta = 7.0

#2.1


In [ ]:
sizes = [5, 10, 100, 200, 400, 600, 800, 1000]
samples_storage = {}
for n in sizes:
    samples_storage[n] = []
    for _ in range(5):
        sample = [laplace(my_mu, my_teta) for _ in range(n)]
        samples_storage[n].append(sample)


print(f"Параметры: mu={my_mu}, teta={my_teta}\n")

for n in sizes:
    print(f"=== Объем выборки n = {n} ===")
    for i in range(5):
        print(f"Выборка №{i+1}:")
        print(samples_storage[n][i]) # Печатает полный список
        print()
    print("-" * 50)

#2.2
Функция для вычисления значения ЭФР в точке x

F_n(x) = (количество элементов <= x) / n

In [ ]:
def laplace_cdf(x, mu, teta):
    if x <= mu:
        return 0.5 * math.exp((x - mu) * teta)
    else:
        return 1 - 0.5 * math.exp(-(x - mu) * teta)

In [ ]:
def get_ecdf_value(sample, x):
    count = 0
    for val in sample:
        if val <= x:
            count += 1
    return count / len(sample)

In [ ]:
# Определяем диапазон для оси X (вокруг mu +/- 3*teta)
# Для mu=12, teta=7 это примерно от -10 до 35
x_min = my_mu - 4 * my_teta
x_max = my_mu + 4 * my_teta
x_points = [x_min + i * (x_max - x_min) / 200 for i in range(201)] # 200 точек для плавности

plt.figure(figsize=(15, 20)) # Большое полотно

# Палитра цветов для 5 выборок
colors_5 = ['blue', 'green', 'orange', 'purple', 'cyan']

for i, n in enumerate(sizes):
    plt.subplot(4, 2, i + 1)

    # Рисуем ЭФР для всех 5 выборок данного объема
    for k in range(5):
        sample = samples_storage[n][k]
        # Считаем Y для графика
        y_values = [get_ecdf_value(sample, x) for x in x_points]

        # Первая выборка получает метку в легенду
        lbl = f'ЭФР выборки (n={n})' if k == 0 else None

        plt.step(x_points, y_values, where='post', label=lbl, color=colors_5[k], alpha=0.5, linewidth=1)

    # Рисуем теоретическую функцию (жирная красная линия)
    y_theor = [laplace_cdf(x, my_mu, my_teta) for x in x_points]
    plt.plot(x_points, y_theor, label='Теоретическая F(x)', color='red', linewidth=2, linestyle='--')

    plt.title(f'Объем n = {n}')
    plt.xlabel('x')
    plt.ylabel('F(x)')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

D_mn

In [ ]:
def calculate_D_mn(sample1, sample2):
    if sample1 == sample2: return 0
    x = sorted(sample1) # Выборка 1 (объем n)
    y = sorted(sample2) # Выборка 2 (объем m)
    n = len(x)
    m = len(y)

    max_diff = 0.0
    for r in range(1, m + 1):
        val_y = y[r-1]
        f1_val = get_ecdf_value(x, val_y)

        diff_plus = abs(r / m - f1_val)

        diff_minus = abs(f1_val - (r - 1) / m)

        current_max = max(diff_plus, diff_minus)
        if current_max > max_diff:
            max_diff = current_max

    coeff = math.sqrt((n * m) / (n + m))
    return coeff * max_diff


In [ ]:
print("\n" + "="*60)
print("Таблица значений статистики D_mn (сравнение 1-й выборки из каждой серии)")
print("="*60)

# Заголовок таблицы
header = f"{'n \\ m':<6} | " + " | ".join([f"{s:<7}" for s in sizes])
print(header)
print("-" * len(header))

# Заполнение таблицы
for n_row in sizes:
    row_cells = [f"{n_row:<6}"]
    for n_col in sizes:
        # Берем ПЕРВУЮ ([0]) выборку из сохраненных для каждого размера
        s1 = samples_storage[n_row][0]
        s2 = samples_storage[n_col][0]

        val = calculate_D_mn(s1, s2)
        row_cells.append(f"{val:.4f} ")
    print(" | ".join(row_cells))

#2.3

In [ ]:
def laplace_pdf(x, mu, teta):
    # f(x) = (theta / 2) * exp( -theta * |x - mu| )
    return (teta / 2.0) * math.exp(-teta * abs(x - mu))

In [ ]:
plt.figure(figsize=(15, 20))

# Диапазон для отрисовки теоретической кривой
# Поскольку ширина распределения ~ 1/teta.
# При teta=7 ширина очень маленькая! (1/7 ≈ 0.14).
# Диапазон +/- 4/teta будет достаточен.
scale = 1.0 / my_teta
x_plot = np.linspace(my_mu - 6*scale, my_mu + 6*scale, 500)
y_pdf = [laplace_pdf(val, my_mu, my_teta) for val in x_plot]

for i, n in enumerate(sizes):
    plt.subplot(4, 2, i + 1)

    # Берем ПЕРВУЮ выборку
    sample = samples_storage[n][0]

    # 1. Строим ГИСТОГРАММУ
    counts, bins, _ = plt.hist(sample, bins=20, density=True,
                               alpha=0.4, color='skyblue', edgecolor='black', label='Гистограмма')

    # 2. Строим ПОЛИГОН ЧАСТОТ
    bin_centers = 0.5 * (bins[:-1] + bins[1:])
    plt.plot(bin_centers, counts, 'b-o', linewidth=2, label='Полигон частот')

    # 3. Строим ТЕОРЕТИЧЕСКУЮ ПЛОТНОСТЬ
    plt.plot(x_plot, y_pdf, 'r--', linewidth=2, label='Плотность f(x)')

    plt.title(f'Объем выборки n = {n}')
    plt.xlabel('x')
    plt.ylabel('Плотность вероятности')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()



#2.4

In [ ]:
import pandas as pd

#  Вычисляет выборочное среднее: sum(x_i) / n
def calculate_mean(sample: list) -> float:
    if not sample: return 0.0
    return sum(sample) / len(sample)
#   Вычисляет выборочную дисперсию: sum((x_i - mean)^2) / n
def calculate_variance(sample: list) -> float:
    n = len(sample)
    if n <= 1: return 0.0

    mean_val = calculate_mean(sample)
    sum_sq_diff = 0.0
    for x in sample:
        sum_sq_diff += (x - mean_val) ** 2

    return sum_sq_diff / n

# Истинные значения
true_mean = my_mu
true_var = 2.0 / (my_teta ** 2)

# Словарь для сбора всех результатов
results = []

for n in sizes:
    for i in range(5):
        sample = samples_storage[n][i]

        # Вычисление выборочного среднего
        sample_mean = calculate_mean(sample)

        # Вычисление выборочной дисперсии (смещенной, делитель n)
        # S^2 = (1/n) * sum((xi - mean)^2)
        sample_var = calculate_variance(sample)
        # Добавляем в таблицу
        results.append({
            "n": n,
            "Выборка": i + 1,
            "Выб. среднее X_bar": sample_mean,
            "Ошибка (X_bar)": abs(sample_mean - true_mean),
            "Выб. дисперсия S^2": sample_var,
            "Ошибка (S^2)": abs(sample_var - true_var)
        })

# Создаем DataFrame для красивого вывода
df_results = pd.DataFrame(results)

# Выводим таблицу
# Для наглядности выведем усредненные ошибки по каждому n
print("\nУсредненные ошибки оценок по 5 выборкам для каждого n:")
print(df_results.groupby("n")[["Ошибка (X_bar)", "Ошибка (S^2)"]].mean())

print("\n" + "="*80)
print("ДЕТАЛЬНАЯ ТАБЛИЦА (первые строки):")
print(df_results.to_string(index=False))
# Если хочешь увидеть всё, убери .head(10)



# 3.1

In [ ]:
import pandas as pd
import numpy as np
import math

# === Задание 3.1: Оценки параметра Theta (только по выборке) ===

def get_theta_mm(sample):
    n = len(sample)
    x_bar = sum(sample) / n
    var_s2 = sum([(x - x_bar)**2 for x in sample]) / n

    if var_s2 > 0:
        return math.sqrt(2 / var_s2)
    else:
        return 0.0

def get_theta_mle(sample):
    n = len(sample)
    # Считаем знаменатель: сумма модулей отклонений от медианы
    abs_diff_sum = sum([abs(x - my_mu) for x in sample])

    if abs_diff_sum > 0:
        return n / abs_diff_sum
    else:
        return 0.0

# --- Расчет и красивый вывод ---

estimates_data = []

for n in sizes:
    for i in range(5):
        sample = samples_storage[n][i]

        # Функции теперь принимают только sample
        val_mm = get_theta_mm(sample)
        val_mle = get_theta_mle(sample)

        estimates_data.append({
            "n": n,
            "Выборка": i + 1,
            "Theta (MM)": val_mm,
            "Theta (MLE)": val_mle
        })

# Формируем таблицу
df_estimates = pd.DataFrame(estimates_data)

# Настраиваем отображение (4 знака после запятой)
pd.options.display.float_format = '{:.4f}'.format

# Выводим полную таблицу
print(df_estimates.to_string(index=False))

#3.2

In [ ]:
def opt(sample, mu = my_mu):
  diff_sum = 0
  for i in sample:
    diff_sum += abs(i - mu)
  return (len(sample) - 1) / diff_sum

In [ ]:
estimates_data = []

for n in sizes:
    for i in range(5):
        sample = samples_storage[n][i]

        # Функции теперь принимают только sample
        val_opt = opt(sample)
        diff = abs(val_opt - my_teta)
        estimates_data.append({
            "n": n,
            "Выборка": i + 1,
            "Theta (opt)": val_opt,
            "Diff (opt - theta)": diff
        })

# Формируем таблицу
df_estimates = pd.DataFrame(estimates_data)

# Настраиваем отображение (4 знака после запятой)
pd.options.display.float_format = '{:.4f}'.format

# Выводим полную таблицу
print(df_estimates.to_string(index=False))

#3.3

In [ ]:
import kagglehub
path = kagglehub.dataset_download("mczielinski/bitcoin-historical-data")

In [ ]:
import os
df = pd.read_csv(os.path.join(path, "btcusd_1-min_data.csv"))
print(df.head())

In [ ]:
# 1. Переводим Timestamp в datetime
df['datetime'] = pd.to_datetime(df['Timestamp'], unit='s')

# 2. Делаем дневной ряд цен закрытия
daily_close = (
    df
    .set_index('datetime')['Close']   # ставим время индексом и берём Close
    .resample('D')                    # группируем по дням
    .last()                           # берём последний Close за день
    .ffill()                          # если есть пропуски дней — тянем предыдущее значение
)

# 3. Создаём df_daily на основе дневных цен
df_daily = daily_close.to_frame(name='Close')

# 4. Считаем лог-нормальные доходности день к дню
df_daily['log_return_daily'] = np.log(df_daily['Close']).diff()
df_daily = df_daily.dropna()
print(df_daily.head())

In [ ]:
ax = df_daily['log_return_daily'].plot(kind='hist', bins=100, density=True)
plt.show()

In [ ]:
sample_list = df_daily['log_return_daily'].tolist()

In [ ]:
mean_real = calculate_mean(sample_list)
var_real = calculate_variance(sample_list)
opt_real = opt(sample_list, mean_real)

print("ЗНАЧЕНИЯ ВЫБОРОЧНЫХ МОМЕНТОВ И ОПТИМАЛЬНОЙ ОЦЕНКИ ДЛЯ РЕАЛЬНЫХ ДАННЫХ")
print(f"Выборочное матожидание: {mean_real:.3f}")
print(f"Выборочная дисперсия: {var_real:.3f}")
print(f"Оптимальная оценка theta: {opt_real:.3f}")

In [ ]:
def laplace_pdf(x, teta, mu):
  return teta/2 * np.exp(-teta*np.abs(x-mu))
# данные
x = df_daily['log_return_daily'].dropna()

# гистограмма (нормированная как плотность)
ax = x.plot(kind='hist', bins=200, density=True, alpha=0.6, label='Empirical')

# координаты для теоретической плотности
xs = np.linspace(x.min(), x.max(), 500)
pdf = laplace_pdf(xs, opt_real, mean_real)

# добавляем теоретическую кривую
plt.plot(xs, pdf, 'r-', lw=2, label=f'Laplace PDF\nμ={mean_real:.4g}, theta={opt_real:.4g}')
plt.xlabel('log_return_daily')
plt.ylabel('Density')
plt.legend()
plt.show()

# 4

Критерий Колмогорова

In [ ]:
def calculate_D(sample, mu=my_mu, teta=my_teta):
    """
    Возвращает D_n = sup_x |F_n(x) - F(x)| для дискретного случая (с повторами).
    """
    n = len(sample)

    x_sorted = sorted(sample)

    D_plus = 0.0   # max (F_n(x-) - F(x))  = max( (i-1)/n - F(x_i) ) по группам [web:46]
    D_minus = 0.0  # max (F(x) - F_n(x))   = max( F(x_i) - i/n )      по группам [web:46]

    i = 0
    while i < n:
        x = x_sorted[i]

        # группа одинаковых значений x: индексы i..j-1
        j = i
        while j < n and x_sorted[j] == x:
            j += 1

        F = laplace_cdf(x, mu, teta)

        # слева от x: F_n(x-) = i/n (т.к. i элементов строго меньше x)
        Fn_left = i / n
        D_plus = max(D_plus, Fn_left - F)

        # справа в x: F_n(x) = j/n (т.к. j элементов <= x)
        Fn_right = j / n
        D_minus = max(D_minus, F - Fn_right)

        i = j

    return max(D_plus, D_minus)

In [ ]:
from scipy.stats import kstwobign
alpha = 0.05

In [ ]:
def kriteriyK(sample, teta=my_teta, mu=my_mu):
  if len(sample) <=20:
    T = (6 * n * calculate_D(sample, mu=my_mu, teta=teta) + 1)/(6*n**0.5)
  else:
    T = n**0.5 * calculate_D(sample, mu=my_mu, teta=teta)
  critical_value = kstwobign.ppf(1 - alpha)
  return T >= critical_value, T, critical_value

In [ ]:
print("\n" + "="*95)
print(f"ТАБЛИЦА КРИТЕРИЯ КОЛМОГОРОВА ДЛЯ ПРОСТОЙ ГИПОТЕЗЫ")
print("="*95)
header = f"{'Объем n':<10} | {'Номер выборки':<15} |{'Вывод':<15} | {'T(X)':<15} | {'T_alpha':<15}"
print(header)
print("-" * len(header))

# Проходим по всем размерам выборок
for n in sizes:
  for sample_number in range(5):
    sample = samples_storage[n][sample_number]
    ans, T, t = kriteriyK(sample)
    if ans:
      ans_str = "H_0 отвергается"
    else:
      ans_str = "H_0 принимается"
    print(f"{n:<10} | {sample_number+1:<15} |{ans_str:<15} | {T:<15.7f} | {t:<15.4f}")
  print("-" * len(header))

In [ ]:
print("\n" + "="*95)
print(f"ТАБЛИЦА КРИТЕРИЯ КОЛМОГОРОВА ДЛЯ СЛОЖНОЙ ГИПОТЕЗЫ")
print("="*95)
header = f"{'Объем n':<10} | {'Номер выборки':<15} |{'Вывод':<15} | {'teta_mle':<15} | {'T(X)':<15} | {'T_alpha':<15}"
print(header)
print("-" * len(header))

# Проходим по всем размерам выборок
for n in sizes:
  for sample_number in range(5):
    half = n//2
    sample1 = samples_storage[n][sample_number][:half]
    sample2 = samples_storage[n][sample_number][half:]
    tet = get_theta_mle(sample1)
    ans, T, t = kriteriyK(sample2, tet)
    if ans:
      ans_str = "H_0 отвергается"
    else:
      ans_str = "H_0 принимается"
    print(f"{n:<10} | {sample_number+1:<15} |{ans_str:<15} | {tet:<15.7f} | {T:<15.7f} | {t:<15.4f}")
  print("-" * len(header))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kstwobign

# x = значения для аргумента предельного распределения sqrt(n) * D_n
x = np.linspace(0.0, 1.0, 1000)
F = kstwobign.ppf(1 - x)  # CDF [web:82]

plt.figure(figsize=(7, 4))
plt.plot(x, F, lw=2)
plt.title("Функция распределения Колмогорова (SciPy: kstwobign)")
plt.xlabel("x")
plt.ylabel("F(x)=P(√n·D_n ≤ x)")
plt.grid(True, alpha=0.3)
plt.show()


## Хи-квадрат


In [ ]:
import numpy as np
from scipy.stats import chi2

alpha = 0.05

In [ ]:
def laplace_quantiles(p, mu, teta):
    if p < 0.5:
        return mu + (1 / teta) * math.log(2 * p)
    else:
        return mu - (1 / teta) * math.log(2 * (1 - p))

def build_v_p(sample, k, mu, teta):
    sample = np.asarray(sample)

    # 1. Генерируем k+1 границ для равновероятных интервалов
    # Вероятности границ: 0, 1/k, 2/k, ..., 1
    probs = np.linspace(0, 1, k + 1)

    # Находим соответствующие x (границы бинов)
    # prob=0 -> -inf, prob=1 -> +inf
    bins = []
    for p_val in probs:
        if p_val == 0:
            bins.append(-np.inf)
        elif p_val == 1:
            bins.append(np.inf)
        else:
            bins.append(laplace_quantiles(p_val, mu, teta))

    bins = np.array(bins)

    # 2. Считаем попавшие в интервалы значения (v)
    v, _ = np.histogram(sample, bins=bins)

    # 3. Считаем теоретические вероятности (p)
    # p[i] = CDF(bins[i+1]) - CDF(bins[i])
    p = []
    for i in range(len(bins) - 1):
        # Левая граница
        if bins[i] == -np.inf:
            cdf_left = 0.0
        else:
            cdf_left = laplace_cdf(bins[i], mu, teta)

        # Правая граница
        if bins[i+1] == np.inf:
            cdf_right = 1.0
        else:
            cdf_right = laplace_cdf(bins[i+1], mu, teta)

        p.append(cdf_right - cdf_left)

    return np.array(v), np.array(p)

In [ ]:
def calculate_Hi2(v, p):
  N = len(v)
  n = sum(v)

  hi2 = 0.0
  for j in range(N):
    hi2 += (v[j] - n * p[j])**2 / (n * p[j])
  return hi2

In [ ]:
def criteriyHi2(sample, k, teta=my_teta, r=0, alpha=alpha, mu=my_mu):
  v, p = build_v_p(sample, k, mu, teta)
  hi2_n = calculate_Hi2(v, p)
  t = chi2.ppf(1 - alpha, k - 1 - r)
  return hi2_n >= t, hi2_n, t

In [ ]:
print("\n" + "="*95)
print(f"ТАБЛИЦА КРИТЕРИЯ ХИ КВАДРАТ ДЛЯ ПРОСТОЙ ГИПОТЕЗЫ")
print("="*95)
header = f"{'Объем n':<10} | {'Номер выборки':<15} | {'Кол. интервалов':<15} |{'Вывод':<15} | {'Х^2_n(X)':<15} | {'T_alpha':<15}"
print(header)
print("-" * len(header))

# Проходим по всем размерам выборок
for n in sizes:
  for count_interval in [3, 5, 10, 30]:
    for sample_number in range(5):
      sample = samples_storage[n][sample_number]
      ans, T, t = criteriyHi2(sample, count_interval)
      if ans:
        ans_str = "H_0 отвергается"
      else:
        ans_str = "H_0 принимается"
      print(f"{n:<10} | {sample_number+1:<15}| {count_interval:<15} |{ans_str:<15} | {T:<15.7f} | {t:<15.4f}")
    print("-" * len(header))
  print("-" * len(header))

In [ ]:
print("\n" + "="*95)
print(f"ТАБЛИЦА КРИТЕРИЯ ХИ КВАДРАТ ДЛЯ СЛОЖНОЙ ГИПОТЕЗЫ")
print("="*95)
header = f"{'Объем n':<10} | {'Номер выборки':<15} | {'Кол. интервалов':<15} |{'Оценка МП':<15} |{'Вывод':<15} | {'Х^2_n(X)':<15} | {'T_alpha':<15}"
print(header)
print("-" * len(header))

# Проходим по всем размерам выборок
for n in sizes:
  for count_interval in [3, 5, 10, 30]:
    for sample_number in range(5):
      sample = samples_storage[n][sample_number]
      tet = get_theta_mle(sample)
      ans, T, t = criteriyHi2(sample, count_interval, tet, 1)
      if ans:
        ans_str = "H_0 отвергается"
      else:
        ans_str = "H_0 принимается"
      print(f"{n:<10} | {sample_number+1:<15} | {count_interval:<15} | {tet:<15.7f} | {ans_str:<15} | {T:<15.7f} | {t:<15.4f}")
    print("-" * len(header))
  print("-" * len(header))

## 4.1

In [ ]:
import math
from scipy.stats import kstwobign  # Распределение Колмогорова

def check_smirnov_homogeneity(sample1, sample2, alpha=0.05):
    """
    Проверяет гипотезу однородности двух выборок (критерий Смирнова).
    Возвращает: (is_rejected, statistic, critical_value)
    """
    n = len(sample1)
    m = len(sample2)

    # 1. Вычисляем статистику (твоя функция уже включает корень!)
    stat = calculate_D_mn(sample1, sample2)

    # 2. Вычисляем критическое значение t_alpha
    # Для распределения Колмогорова K(t_a) = 1 - alpha => t_a = K^(-1)(1 - alpha)
    # В scipy kstwobign.ppf(q) возвращает квантиль для P(S < x) = q
    # Нам нужно P(S > t_a) = alpha => P(S < t_a) = 1 - alpha
    t_alpha = kstwobign.ppf(1 - alpha)

    # Внимание! Есть два варианта сравнения:
    # Асимптотический (для больших n, m): сравниваем stat с t_alpha
    # Точный (для малых n, m): сложнее, обычно используют асимптотику если n,m > 20-25

    # Твое задание ссылается на теорему Смирнова (предельное распределение),
    # значит используем асимптотический критерий.

    is_rejected = stat > t_alpha

    return is_rejected, stat, t_alpha

In [ ]:
print("\n" + "="*80)
print(f"ПРОВЕРКА ГИПОТЕЗЫ ОБ ОДНОРОДНОСТИ (alpha={0.05})")
print("="*80)
header = f"{'Размер n,m':<12} | {'Пара выборок':<15} | {'Статистика':<12} | {'Крит. зн.':<12} | {'Вывод':<15}"
print(header)
print("-" * len(header))

for n in sizes: # size - твой массив размеров [20, 50, ...]
  for m in sizes:
    s1 = samples_storage[n][0]
    s2 = samples_storage[m][1]

    rejected, S, t_crit = check_smirnov_homogeneity(s1, s2)

    res_str = "H0 отвергается" if rejected else "H0 принимается"
    pair_name = f"{n1} vs {m2}"

    print(f"{n:<6}, {m: < 5} | {pair_name:<15} | {S:<12.4f} | {t_crit:<12.4f} | {res_str:<15}")

print("-" * len(header))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Параметры кредита
principal = 2_200_000       # Сумма кредита (руб)
annual_rate = 3.2         # Годовая ставка (%)
years_range = range(1, 21)  # Срок от 1 до 20 лет

# Вспомогательные вычисления
monthly_rate = annual_rate / 12 / 100
payments = []
total_payouts = []

for years in years_range:
    n_months = years * 12
    # Формула аннуитетного платежа
    if monthly_rate > 0:
        annuity = principal * (monthly_rate * (1 + monthly_rate)**n_months) / ((1 + monthly_rate)**n_months - 1)
    else:
        annuity = principal / n_months

    payments.append(annuity)
    total_payouts.append(annuity * n_months)

# Построение графика
fig, ax1 = plt.subplots(figsize=(10, 6))

# Первая ось Y (Ежемесячный платеж) - синяя линия
color = 'tab:blue'
ax1.set_xlabel('Срок кредита (лет)')
ax1.set_ylabel('Ежемесячный платеж (руб)', color=color)
ax1.plot(years_range, payments, color=color, marker='o', label='Ежемесячный платеж')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True)

# Вторая ось Y (Общая сумма выплат) - красная линия
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Общая сумма выплат (млн руб)', color=color)
ax2.plot(years_range, np.array(total_payouts) / 1_000_000, color=color, linestyle='--', marker='s', label='Общая переплата')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Зависимость платежа и переплаты от срока кредита')
plt.show()
